# Sua Primeira Rede Neural Convolucional com PyTorch do Zero

## Hardware e Dispositivo

Ao treinar e utilizar redes neurais, podemos utilizar tanto o processador como a placa de vídeo para operar os cálculos com tensores 4d. De longe, a placa de vídeo é o dispositivo mais adequado, mas nem sempre está disponível. No PyTorch podemos configurar o dispositivo utilizando o método device. O método device deverá receber na maioria do tempo um de dois valores: "cpu" para processador ou "cuda" para placa de vídeo. Para saber se a placa de vídeo pode ser utilizada, chamamos o método is_available de torch.cuda. O método retornara True caso a gpu esteja disponível. Geralmente, selecionamos o dispositivo conforme o exemplo abaixo. Assim, a gpu é sempre usada quando possível.

In [2]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Dispositivo selecionado: {device}")

Dispositivo selecionado: cuda


### Tensores Fictícios

Para fazer algumas operações com pytorch, utilizamos tensores que representam imagens fictícias. Para gerar um tensor fictício, podemos utilizar o método randn do pytorch. O método receberá como parâmetros o tamanho de cada dimensão do tensor. No exemplo abaixo, estamos gerando um tensor 4d no formato (B, C, H, W) contento 16 imagens fictícias coloridas de 224x224.

IMPORTANTE: Sempre que criarmos um tensor no python, o padrão do pytorch é criá-lo no processador, utilizando a memória RAM do sistema. Quando encadeamos o método to, passando como parâmetro o dispositivo, estamos transferindo o tensor para o dispositivo. Nesse exemplo, o tensor nasce no processador e logo depois o transferimos para a VRAM (Memória RAM da GPU).

In [4]:
x = torch.randn(16, 3, 224, 224).to(device)

Em um problema de classificação, precisaremos tanto das imagens quanto do gabarito. Podemos simular um gabarito utilizando o método randint do pytorch. O método randint gerará um tensor de números inteiros. No exemplo abaixo, utilizamos o parâmetro low para definir o valor mínimo, high para definir o valor máximo e size para definir as dimensões do tensor. Nesse caso, uma única dimensão de tamanho 16.

Ao lidar com tensores, geralmente estaremos em uma dentre duas situações: gerando um tensor do zero ou manipulando um tensor já existente. Quando criamos um tensor do zero, podemos definir o dispositivo diretamente na chamada do método utilizando parâmetro device, conforme o exemplo abaixo. No nosso contexto, o tensor de gabarito nascerá diretamente na memória da gpu (VRAM). Quando o tensor já existe e nós não o geramos, só podemos transferí-lo entre as memórias. Então o tensor nasceu no processador e terá de ser transferida para a gpu, conforme necessário. Para tensores minúsculos como estes que estamos gerando, a diferença é negligenciável. Mas para matrizes enormes que podem superar gigabytes de tamanho, o ideal é sempre gerá-las direto na gpu para evitar gargalos desnecessários da memória RAM.

No exemplo, o que estamos fazendo é, em resumo: gerando um tensor com 16 números aleatórios de 0 à 3 diretamente na memória de vídeo (VRAM).

In [5]:
y = torch.randint(low=0, high=3, size=(16, ), device=device)

Eis o resultado. Abaixo, utilizamos a propriedade shape dos tensores para inspecionar o formato deles. Como queríamos, o tensor fictício de imagens tem quatro dimensões com os tamanhos que especificamos e o tensor fictício do gabarito tem uma dimensão de tamanho 16.

In [6]:
print(x.shape)
print(y.shape)

torch.Size([16, 3, 224, 224])
torch.Size([16])


## Rede Sequencial

Nessa segunda etapa, definimos uma rede sequencial utilizando a classe Sequential de torch.nn. Essa classe receberá vários passos em ordem. O primeiro passo que definimos foi um passo de convolução com a classe Conv2d de nn. Essa classe recebe alguns parâmetros como in_channels, que define a quantidade de canais do input. O parâmetro out_channels define a quantidade de canais do output. O parâmetro kernel_size define o tamanho da lente de convolução e stride define o passo. O que vai acontecer nesse primeiro passo é que será aplicada uma convolução para um tensor de 3 canais (RGB) calculando o produto escalar a partir de uma matriz 3x3 ao redor do pixel atual, pulando os pixels de um a um. Em uma convolução, quando utilizamos um kernel 3x3, teremos como resultado uma imagem com 2 pixels a menos por dimensão do que a imagem original. Como nossas imagens são de 224x224, esta convolução retornaria naturalmente 222x222. Para preservar o tamanho da imagem, especificamos um padding que adicionará uma quantidade de pixels nas bordas. Assim, a imagem seguirá preservando o mesmo tamanho. Aqui, entregamos um tensor 4d no formato (16, 16, 224, 224).

Após aplicar uma convolução, utilizamos a classe ReLU de nn. Que trará a não-linearidade para o modelo. Senão, a rede neural seria apenas uma função linear absurdamente grande. O parâmetro inplace define se os bytes serão modificados no mesmo espaço de memória. Se sim, não será necessário alocar memória RAM extra. Seja a memória do processador ou da gpu. Aqui, entregamos um tensor 4d no formato (16, 16, 224, 224).

A classe MaxPool2d dividirá a imagem em janelas, selecionando o maior valor de cada janela. Nesse exemplo, definimos kernel_size como 2. Então as janelas serão de 2x2. Também definimos stride (passo) como 2. Então pularemos de 2 em 2 pixels. Na prática, essa etapa entregará de volta uma imagem de 112x112 contendo apenas os pixels de maior valor da imagem original. Aqui, entregamos um tensor 4d no formato (16, 16, 112, 112).

Tendo as imagens de 112x112 em mãos, passamos elas para uma classe AdaptiveAvgPool2d. Essa classe recebe um parâmetro output_size que define o tamanho das dimensões da resposta. Como é 2d, tem de ser duas dimensões. No exemplo, definimos que a dimensão deve ser 1x1. Então entra 112x112 e sai 1x1. Essa classe calculará uma média de todos os 112x112 pixels da imagem e entregará de volta essa média em tamanho 1x1 (um único valor). Aqui, entregamos um tensor 4d no formato (16, 16, 1, 1).

Em sequência, a classe Flatten receberá o tensor anterior e o transformará em uma matriz (tensor 2d), mantendo o lote intacto. Aqui, entregamos um tensor 2d no formato (16, 16).

Por fim, a classe Linear realiza uma combinação linear dos atributos extraídos. Essa classe receberá um parâmetro in_features que indica o número de entradas que cada imagem entrega. Este parâmetro tem valor condizente com a quantidade de canais da imagem (nesse caso, 16). O parâmetro out_features determina a quantidade de saídas por amostra. Nesse caso, são três saídas, já que temos três classes de classificação. 

In [8]:
import torch.nn as nn

model = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.AdaptiveAvgPool2d(output_size=(1, 1)),
    nn.Flatten(),
    nn.Linear(in_features=16, out_features=3)
).to(device)

Agora, vamos tentar trazer toda a teoria para o código. O coração desse modelo é o primeiro passo: a convolução. No primeiro ciclo, o pytorch atribuirá pesos completamente aleatórios para cada pixel das lentes de convolução. Após isso, virão vários passos de transformação como a função ReLU para garantir a não linearidade e manter apenas o que ativou a lente de convolução. As transformações continuam até chegar ao classificador (nn.Linear). 

Em problemas de classificação, geralmente temos de resumir todos os números dos pixels gerados pelas lentes de convolução a poucos números. É exatamente isso que fizemos usando a classe AdaptiveAvgPool2d, que resumiu cada canal entregue pelas lentes de convolução a um único número. Formando assim um conjunto de 16 números para os 16 canais. Mas, por quê resumir a poucos números? Porque em problemas de classificação, a posição do objetos não interessa. Se gerássemos matrizes maiores como 7x7, ainda preservaríamos alguma noção de posição, o que não é importante para classificar uma imagem. Além disso, imagine o seguinte. A última camada da rede neural terá de assimilar todos os valores que a camada linear recebeu. Então se tivermos matrizes 7x7, serão 49 valores para cada canal. Com 16 canais, ainda é aceitável, totalizando 7 x 7 x 16 = 784 números. Mas, no mundo real, sempre utilizaremos modelos pré-prontos que não utilizam apenas 16 canais, e sim 256 ou 512 canais por exemplo. Assim, a quantidade de números na última camada da rede neural atingirá a proporção de dezenas de milhares de números. O restante da rede neural precisará escalar para atingir esse patamar, criando milhões de parâmetros a mais. Isso torna a rede muito mais pesada e muito mais propensa a decorar a imagem (overfitting).

### Formato da Saída

Para gerar as predições, podemos chamar a função model que é retornada pela classe Sequential. Passando como parâmetro as imagens. Isso devolverá um tensor 2d no qual a primeira camada tem o tamanho de acordo com o número de imagens do lote e a segunda camada contém a quantidade de features que definimos para a saída. Nesse caso, três features.

In [9]:
predictions = model(x)

print(f"Formato de saída: {predictions.shape}")

Formato de saída: torch.Size([16, 3])


Os dados que recebemos como saída são o que chamamos de logits. Os logits são números brutos desconexos que podem variar do infinito negativo ao infinito positivo. Cada classe de classificação terá seu respectivo logit. Como o resultado das predições é uma matriz de formato (16, 3), cada item da primeira camada é o conjunto de pontuações de uma imagem do input. Para cara conjunto de 3 números, o primeiro logit é a pontuação da classe 0. O segundo logit é a pontuação da classe 1 e o terceiro logit é a pontuação da classe 2. Para que esses números tenham algum significado, é preciso passá-los por uma equação de conversão.

In [10]:
print(f"Dados de saída: \n{predictions}\n")

Dados de saída: 
tensor([[ 0.6205, -0.0739,  0.4677],
        [ 0.6208, -0.0696,  0.4693],
        [ 0.6220, -0.0733,  0.4659],
        [ 0.6174, -0.0757,  0.4701],
        [ 0.6159, -0.0738,  0.4685],
        [ 0.6173, -0.0736,  0.4692],
        [ 0.6177, -0.0721,  0.4663],
        [ 0.6193, -0.0741,  0.4702],
        [ 0.6181, -0.0755,  0.4720],
        [ 0.6212, -0.0710,  0.4673],
        [ 0.6169, -0.0740,  0.4676],
        [ 0.6190, -0.0758,  0.4729],
        [ 0.6136, -0.0745,  0.4700],
        [ 0.6195, -0.0743,  0.4658],
        [ 0.6171, -0.0735,  0.4708],
        [ 0.6197, -0.0757,  0.4698]], device='cuda:0',
       grad_fn=<AddmmBackward0>)



## Calculando o Erro

A equação que utilizaremos para dar sentido aos logits é a equação softmax. Em resumo, ela converte aqueles números desconexos para porcentagens. A soma das porcentagens de cada resultado de imagem tem de atingir exatamente 100%. Dessa forma, conseguimos dizer qual classe o modelo acha mais provável de ser a correta.

In [12]:
probabilities = torch.softmax(predictions, dim=1)

print(probabilities)

tensor([[0.4241, 0.2118, 0.3641],
        [0.4236, 0.2124, 0.3640],
        [0.4247, 0.2119, 0.3633],
        [0.4232, 0.2116, 0.3652],
        [0.4229, 0.2122, 0.3649],
        [0.4231, 0.2120, 0.3649],
        [0.4235, 0.2125, 0.3640],
        [0.4235, 0.2117, 0.3648],
        [0.4230, 0.2114, 0.3655],
        [0.4241, 0.2123, 0.3636],
        [0.4233, 0.2121, 0.3646],
        [0.4231, 0.2112, 0.3656],
        [0.4222, 0.2122, 0.3657],
        [0.4242, 0.2120, 0.3638],
        [0.4228, 0.2119, 0.3653],
        [0.4238, 0.2114, 0.3648]], device='cuda:0', grad_fn=<SoftmaxBackward0>)


Na prática, não aplicaremos manualmente a equação softmax. Utilizaremos uma função de perda unificada. Para isso, utilizamos a classe CrossEntropyLoss de nn. É exatamente essa classe que aplica internamente o log-softmax e calcula o valor numérico (escalar) do erro entre as predições e o gabarito real.

Vale lembrar: a função de perda calcula o tamanho do erro. Mais adiante, ao chamarmos loss.backward(), o motor de diferenciação automática do PyTorch (Autograd) calculará as derivadas parciais (a direção para a qual os pesos teriam de ir para aumentar o erro). Por fim, o otimizador ajustará os pesos na direção contrária para diminuir o erro.

In [13]:
criterion = nn.CrossEntropyLoss()

## Otimização

Agora que as predições de um ciclo já foram entregues e a direção para aumento do erro já está calculada, é hora de ajustar os pesos na direção contrária a fim de diminuir o erro.

A class Adam de torch.optim será a responsável exatamente por essa tarefa. Como primeiro parâmetro, ela receberá os parâmetros do modelo que são exatamente os pesos e os viéses. Também podemos especificar um parâmetro lr (learning rate) que definirá a intensidade do passo que o algoritmo de otimização dará na direção contrária ao erro. Passos muito longos pulam o ponto de erro mínimo. Passos muito curtos demoram excessivamente para convergir. O valor padrão do Adam é de 0.001. 

In [15]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Conectando os Pontos

Finalmente, vamos colocar as coisas para funcionar de fato. Primeiro, precisamos informar à rede neural que ela deve entrar em fase de treinamento. Para isso, chamamos o método train do modelo.

Após isso, faremos um laço de repetição. Esse laço repetirá a sequência de todo um ciclo de treinamento. Os ciclos de treinamento são o que chamamos de épocas. Abaixo, definimos que o treinamento ocorrerá em 10 épocas. 

Para cada época, chamamos o método zero_grad do otimizador. Se não fizermos essa chamada, os gradientes dos lotes anteriores se somarão aos novos indefinidamente, até que em um momento, os pesos da rede virem NaN (Não Número). 

Após zerar os gradientes, realizamos as predições e calculamos a perda utilizando a função criterion que geramos a partir de CrossEntropyLoss. Para calcular o erro, passamos como parâmetros os resultados da predição e o gabarito. O resultado do erro será um tensor com um único número que representa a média daquele erro do lote.

Após calcular o erro, chamamos o método backward do tensor de erro. Assim, o motor de diferenciação automática percorre a rede de trás para frente, calculando as derivadas parciais de cada peso da rede. 

Por fim, o método step do otimizador irá ler os resultados das derivadas parciais e aplicar os ajustes nos parâmetros da rede neural.

In [16]:
model.train()

for epoch in range(10):
    optimizer.zero_grad()

    predictions = model(x)
    loss = criterion(predictions, y)

    loss.backward()
    optimizer.step()

    print(f"Época: {epoch + 1}, Perda: {loss.item():.4f}")

Época: 1, Perda: 1.1845
Época: 2, Perda: 1.1764
Época: 3, Perda: 1.1685
Época: 4, Perda: 1.1610
Época: 5, Perda: 1.1538
Época: 6, Perda: 1.1469
Época: 7, Perda: 1.1404
Época: 8, Perda: 1.1342
Época: 9, Perda: 1.1283
Época: 10, Perda: 1.1227


## Arquitetura Modular

### Estrutura de Pastas

Acabou a brincadeira. Até o momento, fizemos apenas um script plano que processa tensores fictícios. É hora de nos aproximarmos um pouco mais do mundo real. Uma das mudanças a partir de agora é respeitar uma arquitetura de projeto robusta. Confira abaixo a estrutura de pastas base.

```shell
├── data                         # Repositório de imagens
├── src
│   ├── deep_learning
│   │   ├── data                 # Leitura e preparo das imagens
│   │   │   ├── __init__.py
│   │   │   ├── dataset.py       # Classe que lê as imagens do disco
│   │   │   └── transforms.py    # Pipelines de pré-processamento e aumento de dados
│   │   ├── engine               # Motor de treinamento
│   │   │   ├── __init__.py
│   │   │   ├── predictor.py     # Função de inferência para classificar fotos
│   │   │   └── trainer.py       # Loops de treino, validação e early stopping
│   │   ├── models               # Modelos de redes neurais
│   │   │   ├── __init__.py
│   │   │   ├── ...
│   │   │   └── ...
│   │   ├── config.py
│   │   └── __init__.py
│   └── __init__.py
├── main.py                      # Script fino que conecta tudo
├── pyproject.toml
└── uv.lock
```

### Variáveis Globais

Vamos começar pelo mais simples: as variáveis globais do projeto. No arquivo config.py, exportaremos diversas constantes que ditam como algumas coisas funcionam. Como por exemplo o diretório raíz do projeto e o diretório dos dados. Estas variáveis definirão globalmente de onde consumiremos os dados necessários para treinar e avaliar a rede neural.

Antes estávamos trabalhando com um único arquivo, então era muito fácil definir o dispositivo de processamento uma única vez e usá-lo. Agora, teremos de ter fácil acesso ao dispositivo. Por isso, definimos uma constante que contém o dispositivo.

Algumas outras configurações que podemos precisar globalmente são: o tamanho dos lotes de imagens; o tamanho de cada imagem em pixels; a velocidade de aprendizado da rede neural; o número de épocas do treinamento.

In [ ]:
from pathlib import Path

import torch

ROOT_DIR = Path(__file__).resolve().parents[2]
DATA_DIR = ROOT_DIR / "data"
TRAIN_DATA_DIR = DATA_DIR / "Train"
TEST_DATA_DIR = DATA_DIR / "Test"
VALIDATION_DATA_DIR = DATA_DIR / "Validation"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

LEARNING_RATE = 0.001
NUM_EPOCHS = 20

### Ingestão de Dados

Antes de começar a receber os dados, eis a estrutura do nosso diretório de dados utilizando o dataset [Plant disease recognition dataset](https://www.kaggle.com/datasets/rashikrahmanpritom/plant-disease-recognition-dataset).

```shell
├── data
│   ├── Test
│   │   ├── Healthy 
│   │   ├── Powdery 
│   │   └── Rust 
│   ├── Train
│   │   ├── Healthy 
│   │   ├── Powdery 
│   │   └── Rust 
│   └── Validation
│       ├── Healthy 
│       ├── Powdery 
│       └── Rust 
└── ...
```

Imagine que temos Gigabytes e Gigabytes de dados. Não seria viável mantê-los todos na memória RAM durante o treinamento e avaliação. Certo? Por esse motivo, implementaremos um mecanismo de lazy loading, no qual as imagens são carregadas sob demanda.

Para criar esse mecanismo, definimos uma classe ImageFolderDataset que herda da classe Dataset do torch. O construtor dessa classe receberá um diretório base de onde deverá carregar as imagens e uma função de transformação opcional. O que o construtor faz é basicamente ler o diretório de dados e preencher uma propriedade samples com tuplas que contém o caminho até a imagem e a classe dela, que assume o nome dos subdiretórios do caminho base. Nesse caso, devem ser: "Healthy", "Powdery" e "Rust". 

In [ ]:
from pathlib import Path
from torch.utils.data import Dataset
from deep_learning.config import DATA_DIR


class ImageFolderDataset(Dataset):
    def __init__(self, root_dir: str | Path = DATA_DIR, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform

        self.classes = sorted(
            [dir.name for dir in self.root_dir.iterdir() if dir.is_dir()]
        )
        self.class_to_index = {cls_name: i for i, cls_name in enumerate(self.classes)}

        self.samples = []

        for cls_name in self.classes:
            folder = self.root_dir / cls_name

            for img_path in folder.glob("*.*"):
                if img_path.suffix.lower() in (".jpg", ".png", ".jpeg"):
                    self.samples.append((img_path, self.class_to_index[cls_name]))

    # ...

Além do construtor, implementaremos os métodos especiais \_\_len\_\_ e \_\_getitem\_\_ que tornam a classe iterável. O primeiro deve retornar o valor de tamanho quando utilizamos a função len. Nesse caso, retorna a quantidade de amostras de imagens que a classe possui. O segundo método recebe um índice, recupera da lista de amostras uma amostra no índice desejado, carrega a imagem utilizando PIL e converte para RGB, aplica a transformação caso haja uma função de transformação para ser aplicada e finalmente retorna tanto a imagem carregada como sua classe.

In [ ]:
from PIL import Image


class ImageFolderDataset(Dataset):
    # ...

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index: int):
        img_path, label = self.samples[index]

        image = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label

#### Transformadores

No arquivo transforms.py é que devemos definir as funções de transformação: uma para cada partição (treino, teste e validação). Como elas devem ser mantidas separadas, definimos três funções que recebem a imagem do PIL, redimensionam proporcionalmente para o tamanho definido nas configurações (Center Crop), convertem para float32 entre 0 e 1, reordenam os eixos de (H, W, C) para (C, H, W) conforme exigido pelas convoluções do PyTorch, e finalmente retornam o Tensor.

In [ ]:
import numpy
import torch
from PIL import Image, ImageOps
from deep_learning.config import IMAGE_SIZE


def train_transform(image: Image.Image) -> torch.Tensor:
    image = ImageOps.fit(image, size=IMAGE_SIZE, method=Image.Resampling.BILINEAR)

    if numpy.random.rand() < 0.5:
        angle = float(numpy.random.uniform(-15.0, 15.0))
        image = image.rotate(angle=angle)

    if numpy.random.rand() < 0.5:
        image = image.transpose(Image.FLIP_LEFT_RIGHT)

    tensor_numpy = numpy.array(image, dtype=numpy.float32) / 255
    tensor_numpy = tensor_numpy.transpose(2, 0, 1)

    return torch.from_numpy(tensor_numpy)


def test_transform(image: Image.Image) -> torch.Tensor:
    image = ImageOps.fit(image, size=IMAGE_SIZE, method=Image.Resampling.BILINEAR)

    tensor_numpy = numpy.array(image, dtype=numpy.float32) / 255
    tensor_numpy = tensor_numpy.transpose(2, 0, 1)

    return torch.from_numpy(tensor_numpy)


def validation_transform(image: Image.Image) -> torch.Tensor:
    image = ImageOps.fit(image, size=IMAGE_SIZE, method=Image.Resampling.BILINEAR)

    tensor_numpy = numpy.array(image, dtype=numpy.float32) / 255
    tensor_numpy = tensor_numpy.transpose(2, 0, 1)

    return torch.from_numpy(tensor_numpy)

##### Aumentação dos Dados

Se você prestou muita atenção ao ler, certamente percebeu que a função train_transform é um tanto diferente das outras funções. Isso acontece por um motivo muito simples. Quanto mais dados temos, mais sinais conseguimos identificar e mais inteligente será o modelo. A grande sacada é que conseguimos aumentar a quantidade de dados sem necessariamente conseguir imagens completamente novas. É possível fazer aumentação de dados com base apenas nas imagens existentes.

Os filtros de convolução não são sensíveis à translação. Onde quer que o padrão esteja na imagem, a rede neural conseguirá identificá-lo. Mas, eles são sensíveis à rotação e à transposição. O que significa que rotacionar uma imagem ou espelhá-la é uma forma válida de aumentar os dados. Afinal, o filtro de convolução não é capaz de rotacionar e nem de deslizar da direita para a esquerda. É uma caixa rígida que desliza apenas da esquerda para a direita. Então olhando exatamente para a mesma mancha espelhada ou levemente rotacionada, os filtros de convolução gerarão novos sinais.

Não há motivo em aumentar os dados de teste e validação, por isso, a aumentação dos dados geralmente é realizada apenas nos dados de treino. 

Portanto, a função train_transform faz o seguinte: com 50% de chance, rotaciona a imagem em um ângulo entre 15 graus negativos e 15 graus positivos (-15° a +15°). Com outros 50% de chance, espelha a imagem da esquerda para a direita.

Ao rotacionar a imagem, é importante não exagerar no ângulo. Quanto maior for o ângulo, mais a imagem será tomada por cantos pretos. É importante evitar que estes cantos tomem um espaço excessivo da imagem.

In [ ]:
# ...

from PIL.Image import Transpose


def train_transform(image: Image.Image) -> torch.Tensor:
    if numpy.random.rand() < 0.5:
        angle = float(numpy.random.uniform(-15.0, 15.0))
        image = image.rotate(angle=angle)

    if numpy.random.rand() < 0.5:
        image = image.transpose(Transpose.FLIP_LEFT_RIGHT)

    image = ImageOps.fit(image, size=IMAGE_SIZE, method=Image.Resampling.BILINEAR)

    tensor_numpy = numpy.array(image, dtype=numpy.float32) / 255
    tensor_numpy = tensor_numpy.transpose(2, 0, 1)

    return torch.from_numpy(tensor_numpy)

#### Data Loaders

Voltando ao dataset.py, definimos uma instância de ImageFolderDataset para cada etapa. Seja de treino, teste ou validação. Aplicando as respectivas funções de transformação.

In [ ]:
import torch
from torch.utils.data import DataLoader

from deep_learning.config import (
    BATCH_SIZE,
    DATA_DIR,
    TEST_DATA_DIR,
    TRAIN_DATA_DIR,
    VALIDATION_DATA_DIR,
)
from deep_learning.data.transforms import (
    test_transform,
    train_transform,
    validation_transform,
)

# ...

train_dataset = ImageFolderDataset(root_dir=TRAIN_DATA_DIR, transform=train_transform)
test_dataset = ImageFolderDataset(root_dir=TEST_DATA_DIR, transform=test_transform)
validation_dataset = ImageFolderDataset(root_dir=VALIDATION_DATA_DIR, transform=validation_transform)

# ...

Por fim, definimos um data loader para cada etapa utilizando a classe DataLoader do torch. Essa é a classe responsável por prover os dados em lote a partir da classe iterável que construímos. O parâmetro dataset deve receber exatamente o dataset iterável. O parâmetro batch_size define o tamanho do lote. O shuffle define se os dados deverão ser embaralhados ou não. No treinamento, importa que sejam embaralhados. Afinal, nas classes de dataset, as imagens estão perfeitamente ordenadas por tipo. É importante que a rede neural receba diferentes classificações de forma desordenada para garantir a melhor extração de sinais.

Por padrão, o python executa tudo de forma sequencial no processo principal. Então durante o treinamento, a GPU teria de ficar ociosa esperando a CPU carregar mais imagens do disco. Para evitar esse gargalo, definimos o parâmetro num_workers que cria processos paralelos na CPU para pré-carregar os próximos lotes.

Quando estamos utilizando uma GPU dedicada ligada via barramento PCIe, é necessário ativar o parâmetro pin_memory com valor True. A memória RAM convencional do sistema não permite que os bytes sejam transferidos diretamente para a VRAM via DMA a menos que esteja marcada como pinned (travada/page-locked). Portanto, caso a GPU esteja disponível para uso, ativamos este parâmetro.

In [ ]:
# ...

train_data_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_data_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

validation_data_loader = DataLoader(
    dataset=validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

### Definição da Rede Neural

É hora de definir a rede neural que consumirá os dados. Na pasta models, criamos um arquivo com o nome simple_cnn.py. Nele, teremos uma classe SimpleCNN que herda da classe Module de torch.nn. Essa classe terá dois principais métodos: O método construtor e um método forward. O método construtor receberá o número de canais do input e o número de features de output. Também chamará o contrutor da superclasse. Já o método forward receberá um tensor e retornará um tensor. 

Mas, qual a função de cada um? O construtor vai apenas definir e configurar as etapas e as camadas da rede neural. Já o forward, receberá um tensor de input para a rede neural e percorrerá todo o caminho pelas etapas que configuramos no construtor.

In [ ]:
import torch
import torch.nn as nn


class SimpleCNN(nn.Module):
    def __init__(self, in_channels: int = 3, num_classes: int = 3):
        super().__init__()
        
        # ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ...

Vamos começar definindo as camadas da rede neural. Mas antes, vamos lembrar o que estamos estudando: deep learning. Vamos ainda mais fundo. O que significa deep learning? Aprendizado profundo, certo? O quê é algo profundo? Algo que é composto por várias camadas. O coração do deep learning é a rede neural. Logo, é exatamente ela que tem de ser profunda e composta por várias camadas. Agora, qual é o diferencial das redes neurais atual frente às antigas que falharam miseravelmente? As convoluções. Portanto, essas múltiplas camadas tem de envolver convoluções. Então o que vamos configurar na rede neural agora é exatamente isso: uma série de convoluções.

Antes de nos aprofundarmos na função de cada coisa. Leia atentamente o código de definição da rede neural abaixo. Perceba o que acontece com os tensores a cada camada.

Percebeu que as camadas fazem quase exatamente a mesma coisa? E que a cada camada de convolução, o número de canais aumenta enquanto a resolução da imagem diminui? Importa que seja dessa forma por um motivo muito simples. Estamos resolvendo um problema de classificação. Então, relembrando: não importa se o padrão está na esquerda da imagem, na direita, em cima, em baixo, em um canto ou no outro. Importa apenas se o padrão está na imagem. Ter uma imagem em alta resolução traz muita precisão sobre onde o padrão está localizado, mas não é relevante para reconhecer o padrão. O que é mais relevante para reconhecer um padrão é o número de canais. 

Pense nos canais como formas diferentes de enxergar a imagem. No início a imagem tem 3 canais. Então são 3 formas diferentes de enxergar. Ainda no início, entregamos 32 canais. Já são mais 29 formas diferentes de enxergar a imagem. Na segunda camada de convolução, transformamos esses 32 canais em 64 canais. Assim, temos mais 32 formas diferentes de enxergar a imagem. Na terceira e última camada de convolução, dobramos novamente a quantidade de canais, atingindo 128. Com isso, adicionamos mais 64 formas de enxergar a imagem. Cada uma dessas formas pode ser importante para reconhecer o padrão. Quem decide quais formas são mais ou menos importantes é a rede neural durante o treinamento. O que nós fazemos é apenas configurar a rede para conseguir extrair mais sinais dos dados.

Mas, porque não podemos dobrar o número de canais preservando a resolução da imagem? Bom, podemos, mas não é muito aconselhável. Imagine o seguinte. Na primeira camada, recebemos a imagem com resolução de 224x224 com 3 canais. Então temos aproximadamente 150.000 valores numéricos. Vamos supor que preservaremos a resolução, aumentando o número de canais. Na última camada, teríamos 224 x 224 x 128 valores numéricos por imagem. Isso resulta em mais de 6.4 milhões de valores para cada imagem. Mas não passamos uma imagem por vez na rede neural, não é? Passamos nesse momento 32 imagens. Então teríamos quase 205 milhões de valores numéricos passando pela rede neural a cada lote. E esses valores estariam todo morando na memória RAM caso o processamento aconteça no processador ou na memṕoria de vídeo, caso aconteça na GPU. O grande ponto é: para problemas de classificação, estaríamos escalando absurdamente o uso de recursos para preservar a imagem. Mas isso não traria benefício.

Outra coisa que podemos nos perguntar é: por quê não podemos aplicar uma única camada de convolução que já diminua a resolução da imagem e aumente a quantidade de canais de uma vez? Bom, por muito tempo, consideramos as camadas centrais das redes neurais como verdadeiras caixas pretas. Mas, houve um estudo que foi capaz de compreender parte do que acontece nelas. Nesse estudo, a primeira camada foi capaz de enxergar linhas e traços. A segunda juntava duas linhas para enxergar ângulos e texturas. A terceira camada juntada texturas para enxergar padrões complexos como a mancha de ferrugem em uma folha. Então essa é a importância de haver várias camadas de convolução. Assim, permitimos que a rede neural entenda os dados passo a passo. E pense em outro ponto: se tivéssemos uma única camada de convolução, o aprendizado não seria profundo, concorda?

Bom. Agora que já sabemos o que tem de ser feito, vamos por a mão na massa. Dividimos a sequência em três blocos. No primeiro, aplicamos uma convolução que recebe três canais e devolve 32 canais. Sempre utilizando o padrão de tamanho da lente 3x3 com padding de 1 pixel. Depois, aplicamos uma função de normalização 2d utilizando a classe nn.BatchNorm2d, que receberá o número de features (canais) como parâmetro. Após a normalização, utilizamos a função ReLU para introduzir a não-linearidade a por fim MaxPool2d para reduzir as dimensões da imagem. Repetimos mais dois ciclos nesse padrão. A cada ciclo, o número de canais de input da convolução será o número de canais de output da convolução anterior. O número de canais de output da convolução deverá ser o dobro do número de canais do input. Sempre dobramos o número de canais e cortamos a resolução pela metade.

Nessa etapa, vimos uma nova função, a BatchNorm2d. Ela normaliza as ativações de cada canal para média 0 e desvio 1. Isso impede que os números escalem descontroladamente com a profundidade da rede, eliminando instabilidades e permitindo um treinamento muito mais rápido e suave. Idealmente, essa função também deve ser utilizada a cada convolução.

In [ ]:
# ...

class SimpleCNN(nn.Module):
    def __init__(self, in_channels: int = 3, num_classes: int = 3):
        # ...
        
        self.features = nn.Sequential(
            # (B, 3, 224, 224) -> (B, 32, 112, 112)
            nn.Conv2d(
                in_channels=in_channels, out_channels=32, kernel_size=3, stride=1, padding=1
            ),
            nn.BatchNorm2d(num_features=32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # (B, 32, 112, 112) -> (B, 64, 56, 56)
            nn.Conv2d(
                in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1
            ),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # (B, 64, 56, 56) -> (B, 128, 28, 28)
            nn.Conv2d(
                in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1
            ),
            nn.BatchNorm2d(num_features=128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # ...


Depois de definirmos a sequência de convoluções da rede neural, podemos definir o que acontece entre as convoluções e o classificador. Este será o gap. O gap aplica uma única função: A AdaptiveAvgPool2d, que lembrando: calcula a média dos valores dos pixels da imagem, entregando o resultado nas dimensões definidas pelo parâmetro output_size. 

Por último, definimos o classifier. Que irá utilizar a função Flatten para fazer a ponte entre o mundo 4d (convoluções) e o mundo 2d (camada linear). A primeira camada linear diminui a quantidade de canais, combinando eles para formar sinais mais valiosos. Novamente, aplicamos a função ReLU para zerar os valores negativos e introduzir a não-linearidade. A função Dropout desliga uma quantidade de neurônios da camada de acordo com o parâmetro p que define a proporção de 0 até 1 de neurônios desligados. Se temos 64 features nessa camada, serão desligados 25 dos 64 neurônios e os outros 39 terão de seguir e inferir um resultado. Em outra situação, 25 neurônios diferentes serão desligados e outros 39 terão de seguir e inferir um resultado. Então a conclusão da rede neural nunca vai depender de neurônios específicos para inferir um resultado, o que causaria overfitting. O modelo decoraria o treino. Por fim, a última camada linear combinará as features restantes, calculando os logits de resultado para o número de classes.

In [ ]:
import torch
import torch.nn as nn


class SimpleCNN(nn.Module):
    def __init__(self, in_channels: int = 3, num_classes: int = 3):
        # ...

        self.gap = nn.AdaptiveAvgPool2d(output_size=(1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=128, out_features=64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(in_features=64, out_features=num_classes)
        )

    # ...


O método forward da classe de definição da rede neural vai apenas integrar todos os componentes anteriores. Ela receberá um tensor, passará o tensor pelas camadas convolucionais. Depois, passará pelo gap, calculando as médias dos canais e por fim passará pelo classificador, retornando os logits de resultado.

In [ ]:
# ... 

class SimpleCNN(nn.Module):
    # ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.gap(x)

        logits = self.classifier(x)

        return logits


### Motor de Treinamento

É hora de começarmos a unir as peças. Nesse estágio, estaremos construindo funções que compõe o motor de treinamento no arquivo trainer.py. A primeira função é a função train_one_epoch que é responsável por executar todo o ciclo de uma época do treinamento da rede neural. Ela receberá como parâmetros o modelo, o data loader, o avaliador, o otimizador e o dispositivo que deve ser utilizado. E retornará uma dupla com dois decimais que informam a perda e a acurácia da época.

In [ ]:
from collections.abc import Callable
import torch
import torch.nn as nn
from torch.utils.data import DataLoader


def train_one_epoch(
    model: nn.Module,
    data_loader: DataLoader[tuple[torch.Tensor, torch.Tensor]],
    criterion: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> tuple[float, float]:
    # ...


Sem o cálculo de perda e acurácia da época, a função se torna muito mais simples. Informamos ao modelo que ele deve entrar em modo de treinamento e iteramos o data loader de imagens que conforme definimos, retorna uma tupla com a imagem e a respectiva classe. Iterando as imagens e suas respectivas classes, enviamos os tensores para o dispositivo que estamos utilizando. E aí entra o ciclo clássico de treinamento. Primeiro chamamos zero_grad do otimizador para não misturar os gradientes entre diferentes lotes de imagens, geramos os logits a partir das imagens passando elas pela rede neural, calculamos a perda com a função criterion a partir dos logits e dos resultados esperados (targets), calculamos o quanto os pesos tem de ser ajustados para ir em direção à diminuição do erro chamando loss.backward e finalmente ajustamos os pesos de acordo, chamando o método step do otimizador. Para o treinamento, isso já bastaria.

In [ ]:
def train_one_epoch(
    # ...
) -> tuple[float, float]:
    model.train()

    for images, targets in data_loader:
        images, targets = images.to(device), targets.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

Mas, é interessante observamos a progressão da rede neural ao passar das épocas. Para isso, podemos calcular a perda média e a acurácia da época. Definimos variávels para sumarizar a perda, o número de acerto e total de amostras da época. A cada lote de imagens, multiplicamos a perda média do lote pelo número de imagens. Obtendo assim a perda total do lote. Então extraímos a classe de maior ativação dentre os logits usando torch.max e somamos à contagem de predições corretas o número de predições que batem com o gabarito (targets). Para terminar a iteração, incrementamos o total de amostras de acordo com o número de imagens do lote. No fim de tudo, fora da repetição, com todas as sumarizações feitas, retornamos a média geral de perda do lote por amostra e a acurácia calculada de acordo com a quantidade de acertos e total de amostras da época. E retornamos ambos.

In [ ]:
def train_one_epoch(
    # ...
) -> tuple[float, float]:
    model.train()
    
    running_loss = 0.0
    correct = 0
    total = 0

    for images, targets in data_loader:
        # ...

        running_loss += loss.item() * images.size(0)

        _, preds = torch.max(logits, dim=1)

        correct += (preds == targets).sum().item()
        total += targets.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

No mesmo arquivo, definimos uma função evaluate. Essa função receberá o modelo, o data loader, o avaliador e o dispositivo. Assim como a função train_one_epoch, ela também calcula a perda e a acurácia de todos os lotes. Mas, com uma diferença. Essa função põe a rede neural em modo de inferência chamando o método eval, travando o Dropout e as médias da normalização. Enquanto a escrita dos gradientes é interrompida com torch.no_grad, geramos predições para todas as imagens do data loader.

Sabemos o que a função faz, agora, vamos pensar um pouco em para quê precisamos dela. Durante o treinamento, chamamos optimizer.step. Esse método tem um único objetivo que é minimizar o erro das imagens do lote atual. Ou seja, o trabalho dele é melhorar a acurácia com base nos dados dos lotes de treino. Por isso, naturalmente tem uma taxa de acerto maior durante o treinamento do que nos testes com imagens que a rede neural nunca viu antes.

Para nos dar um norte do progresso real do treinamento, o que fazemos é combinar quatro fatores. Primeiro, colocamos a rede neural em modo de evaluation, que desativa alteração das estatísticas internas da normalização e desativa o dropout. Utilizando torch.no_grad, impedimos o pytorch de alocar memória para rastrear as operações. Essa combinação já acaba com qualquer rastro matemático residual que poderia influenciar o aprendizado. Mas, além do rastro residual matemático, o que faz a rede neural aprender é essencialmente a combinação de loss.backward e optimizer.step. Por isso, essas duas chamadas ficam completamente de fora nessa etapa de evaluation.

Por um momento, congelamos a capacidade da rede neural de aprender e enquanto isso realizamos predições utilizando os dados de validação e calculamos as mesmas métricas de antes com base nos resultados dessas predições. Como a rede neural nunca aprendeu nada com os dados de validação, os dados sempre são tecnicamente novos e ela nunca poderia ter "decorado" nada desses dados. Por isso, realisticamente falando, a perda e a acurária em validação são métricas mais confiáveis nesse momento.

A diferença entre a acurácia de treino e a acurácia em validação pode apontar para informações importantes. Por exemplo, se a acurácia de treino é absurdamente alta e a acurácia de validação não, pode indicar que a rede neural está sofrendo de overfitting. O mesmo valeria para a perda.

In [ ]:
def evaluate(
    model: nn.Module,
    data_loader: DataLoader,
    criterion: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
    device: torch.device,
):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, targets in data_loader:
            images, targets = images.to(device), targets.to(device)
            logits = model(images)
            loss = criterion(logits, targets)

            running_loss += loss.item() * images.size(0)
            
            _, preds = torch.max(logits, dim=1)

            correct += (preds == targets).sum().item()
            total += images.size(0)

    eval_loss = running_loss / total
    eval_acc = correct / total

    return eval_loss, eval_acc

### Conectando Todas as Peças

Finalmente, no arquivo main.py, juntaremos tudo que foi feito até agora para fazer a rede neural trabalhar de fato. No código abaixo, instanciamos a classe da rede neural que configuramos e transferimos ela para o dispositivo que deve ser utilizado. Também instanciamos o avaliador (nn.CrossEntropyLoss) e o otimizador (torch.optim.Adam) contendo os parâmetros necessários. Após conseguirmos todas as peças que precisamos, fazemos um laço de repetição que se repita para cada época. Nesse laço, chamamos em primeiro lugar a função train_one_epoch e depois a função evaluate, passando os respectivos parâmetros. Além de é claro, exibir a perda e a acurácia parcial de cada etapa.

In [ ]:
import torch
from torch import nn

from deep_learning.config import DEVICE, LEARNING_RATE, NUM_EPOCHS
from deep_learning.data.dataset import train_data_loader, validation_data_loader
from deep_learning.engine.trainer import evaluate, train_one_epoch
from deep_learning.models.simple_cnn import SimpleCNN

model = SimpleCNN().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)

for epoch in range(NUM_EPOCHS):
    epoch_loss, epoch_acc = train_one_epoch(
        model,
        data_loader=train_data_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE,
    )

    eval_loss, eval_acc = evaluate(
        model, 
        data_loader=validation_data_loader, 
        criterion=criterion, 
        device=DEVICE
    )

    print(f"Epoch {epoch + 1}".center(30, "="))
    print(f"Training Loss / Accuracy: {epoch_loss:.4f} / {epoch_acc:.2%}")
    print(f"Validation Loss / Accuracy: {eval_loss:.4f} / {eval_acc:.2%}")

Observe o resultado abaixo das três primeiras épocas do treinamento. Perceba como nas primeiras épocas a acurácia de treino geralmente é muito maior que a acurácia de validação. Em um determinado momento, a acurácia de treinamento e a acurácia de validação se tornam muito similares. Isso indica que a rede neural está de fato aprendendo padrões relevantes.

```shell
Epoch 1
Loss: 0.7761
Accuracy: 71.86%
Eval 1
Loss: 1.0833
Accuracy: 40.00%
Epoch 2
Loss: 0.5631
Accuracy: 80.18%
Eval 2
Loss: 0.8325
Accuracy: 65.00%
Epoch 3
Loss: 0.4682
Accuracy: 83.06%
Eval 3
Loss: 0.5650
Accuracy: 83.33%
```

### Salvando o Modelo

Durante o treinamento da rede neural, a acurácia de validação naturalmente oscila para cima ou para baixo, dependendo por exemplo de variáveis como a aumentação dos dados. Por isso, durante o treinamento, monitoramos a melhor pontuação de perda do modelo. A cada época de treinamento e validação, verificamos se na época atual a perda foi menor do que a menor registrada anteriormente. Caso sim, salvamos o modelo. Mas na realidade não é o modelo que está sendo salvo, e sim os pesos da rede neural. Para isso, utilizamos o método save do torch. Que recebe como primeiro parâmetro o quê deve ser salvo e como segundo parâmetro a localização onde deve ser salvo. Para acessar os pesos da rede neural, utilizamos o método state_dict do modelo. Na prática, ao final de todo o treinamento, o modelo salvo será o melhor de todas as épocas considerando a pontuação de perda na validação.

In [ ]:
from deep_learning.config import MODEL_PATH

# ...

best_loss: float | None = None

for epoch in range(NUM_EPOCHS):
    # ...

    if best_loss is None or eval_loss < best_loss:
        best_loss = eval_loss

        torch.save(model.state_dict(), MODEL_PATH)

    # ...

### Testando o Modelo

Quando testamos a rede neural treinada, testamos também no arquivo main.py. Isso faz com que o arquivo tenha duas funções distintas: treinar e testar. Por isso, é sensato encapsular o treinamento do modelo em uma função a parte, conforme o código abaixo.

In [ ]:
# ...

def train_model(
    model: nn.Module,
    criterion: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
    device: torch.device,
):
    optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)

    best_loss: float | None = None

    for epoch in range(NUM_EPOCHS):
        epoch_loss, epoch_acc = train_one_epoch(
            model,
            data_loader=train_data_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
        )

        eval_loss, eval_acc = evaluate(
            model,
            data_loader=validation_data_loader,
            criterion=criterion,
            device=device,
        )

        if best_loss is None or eval_loss < best_loss:
            best_loss = eval_loss

            torch.save(model.state_dict(), MODEL_PATH)

        print(f" Epoch {epoch + 1} ".center(60, "="))
        print(f"Training Loss / Accuracy: {epoch_loss:.4f} / {epoch_acc:.2%}")
        print(f"Validation Loss / Accuracy: {eval_loss:.4f} / {eval_acc:.2%}")

# ...

Agora, vamos implementar a função de teste do modelo, que também é muito simples. Apenas recebe o modelo a ser testado, o avaliador e o dispositivo. Para testar, reutiliza a função evaluate, passando como data_loader o data loader do dataset de testes. Por fim, exibe a pontuação de perda e a acurácia.

Para testar o modelo, utilizamos o dataset de testes. Este, a rede neural nunca viu durante o treinamento nem para ajustar os pesos e nem como validação parcial. A avaliação ideal de uma rede neural deve ser exatamente assim. Dividida em duas partes: parcial (a cada época) e final (após todas as épocas).

In [ ]:
def test_model(
    model: nn.Module,
    criterion: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
    device: torch.device,
):
    test_loss, test_acc = evaluate(
        model, data_loader=test_data_loader, criterion=criterion, device=device
    )

    print(" Final Validation ".center(60, "="))
    print(f"Training Loss / Accuracy: {test_loss:.4f} / {test_acc:.2%}")


# ...

Adicionamos uma condicional que verifica se o arquivo atual está sendo executado como um programa. Quando estiver, instancia o modelo, o avaliador e o otimizador. E chama função train_model para treinar o modelo.

In [ ]:
# ...

if __name__ == "__main__":
    model = SimpleCNN().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    train_model(model, criterion=criterion, device=DEVICE)

    # ...

Com o modelo treinado, instanciamos outra rede neural exatamente igual. Mas limpa, sem os pesos calculados no treinamento. Depois, utilizamos o método load_state_dict para carregar pesos na rede neural. Como primeiro parâmetro, o método deve receber os pesos. Podemos recuperar os pesos salvos utilizando torch.load, que recebe como primeiro parâmetro a localização onde os pesos estão salvos. Além disso, também recebe um parâmetro map_location que deve apontar para o dispositivo para o qual os pesos devem ser carregados. Para terminar, chamamos a função test_model com os respectivos parâmetros.

In [ ]:
# ...

if __name__ == "__main__":
    # ...

    trained_model = SimpleCNN().to(DEVICE)
    trained_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))

    test_model(trained_model, criterion=criterion, device=DEVICE)

```shell
===================== Final Validation =====================
Training Loss / Accuracy: 0.3545 / 89.33%
```

### Função de Inferência para Produção

No arquivo predictor.py, definiremos uma função predict_single_image e uma função transform. Começando pela função mais simples, que é a transform, ela receberá um tensor do numpy e retornará de volta um tensor do numpy. Essa função realiza a transposição do que deverá ser os canais da imagem, garante a contiguidade dos bytes nas "gavetas" da memória, converte para o tipo float32 e divide por 255, normalizando os dados do tensor.

In [ ]:
import numpy

def transform(tensor: numpy.ndarray) -> numpy.ndarray:
    tensor = tensor.transpose(2, 0, 1)
    tensor = numpy.ascontiguousarray(tensor)
    tensor = tensor.astype(numpy.float32)
    tensor = tensor / 255
                   
    return tensor

Já a função predict_single_image receberá tudo que precisa para realizar a predição, retornando a classe predita e o percentual de certeza. Não vamos nos extender explicando o que cada linha de código faz para não ser repetitivo. Em resumo, a função valida o número de dimensões do tensor recebido, transforma ele usando a função determinada, expande as dimensões para se tornar um tensor 4d, transfere para o dispositivo definido, realiza a predição enquanto desabilita a escrita dos gradientes, aplica a função softmax para converter os logits para probabilidades, encontra a maior probabilidade e seu respectivo índice, seleciona na lista de classes o índice de resultado e retorna exatamente a classe predita com o percentual de certeza que a rede neural teve.

In [ ]:
# ...

from collections.abc import Callable

import torch
from torch import nn

def predict_single_image(
    model: nn.Module,
    classes: list[str],
    device: torch.device,
    tensor: numpy.ndarray,
    transform: Callable[[numpy.ndarray], numpy.ndarray] = transform,
):
    if tensor.ndim != 3:
        raise ValueError(f"Expected 3D tensor. Received {tensor.ndim}D")

    tensor = transform(tensor)
    tensor = numpy.expand_dims(tensor, axis=0)
    tensor = torch.from_numpy(tensor).to(device)

    model.eval()

    with torch.no_grad():
        logits = model(tensor)
        probabilities = torch.softmax(logits, dim=1)
        higher_prob, higher_prob_index = torch.max(probabilities, dim=1)

    return {
        "predicted_class": classes[higher_prob_index.item()],
        "certainty": round(higher_prob.item(), 4),
    }



Abaixo, um exemplo de predição utilizando a função definida.

In [ ]:
from deep_learning.config import IMAGE_SIZE
from PIL import ImageOps
from deep_learning.config import MODEL_PATH
from deep_learning.config import DEVICE
from deep_learning.models.simple_cnn import SimpleCNN
from PIL import Image
import numpy
import torch

model = SimpleCNN().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))

print(
    predict_single_image(
        model,
        classes=["Healthy", "Powdery", "Rust"],
        device=DEVICE,
        tensor=numpy.array(
            ImageOps.fit(
                Image.open(
                    ".../path/to/image.jpg"
                ).convert("RGB"),
                IMAGE_SIZE,
            )
        ),
    )
)